In [33]:

from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

RANDOM_STATE = 42
TARGET = "Churn"

print("Imports completed.")

Imports completed.


In [35]:
df = pd.read_csv(r"C:\Users\KIIT\Desktop\RetailMind\data\WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(r"C:\Users\KIIT\Desktop\RetailMind\data\WA_Fn-UseC_-Telco-Customer-Churn.csv", df.shape)

display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nChurn distribution:")
display(df[TARGET].value_counts())

print("\nChurn percentage:")
display(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)

C:\Users\KIIT\Desktop\RetailMind\data\WA_Fn-UseC_-Telco-Customer-Churn.csv (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


Churn distribution:


Churn
No     5174
Yes    1869
Name: count, dtype: int64


Churn percentage:


Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

In [36]:
# Convert TotalCharges into numeric format.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce",
)

# Remove customer identifier.
df = df.drop(
    columns=["customerID"],
    errors="ignore",
)

# Encode target.
df[TARGET] = df[TARGET].map({
    "No": 0,
    "Yes": 1,
})

if df[TARGET].isna().any():
    raise ValueError("Unexpected values found in Churn.")

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

print("\nMissing values after conversion:")
display(X.isnull().sum()[X.isnull().sum() > 0])

Features: (7043, 19)
Target: (7043,)

Missing values after conversion:


TotalCharges    11
dtype: int64

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn rate:", round(y_train.mean(), 3))
print("Testing churn rate:", round(y_test.mean(), 3))

Training features: (5634, 19)
Testing features: (1409, 19)

Training churn rate: 0.265
Testing churn rate: 0.265


In [38]:
def make_preprocessor(input_frame):

    numeric_columns = (
        input_frame
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    categorical_columns = (
        input_frame
        .select_dtypes(exclude="number")
        .columns
        .tolist()
    )

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            ),
        ]
    )

    return preprocessor


def make_model_pipeline(input_frame, estimator):

    return Pipeline(steps=[
        ("preprocess", make_preprocessor(input_frame)),
        ("model", estimator),
    ])


print("Preprocessing functions created.")

Preprocessing functions created.


In [39]:
lr_options = {
    "Unweighted": None,
    "Balanced": "balanced",
}

cv_strategy_day1 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

day1_trials = {}

for option_name, weight in lr_options.items():

    print("Testing:", option_name)

    candidate = make_model_pipeline(
        X_train,
        LogisticRegression(
            max_iter=2000,
            class_weight=weight,
            random_state=RANDOM_STATE,
        ),
    )

    # Out-of-fold probabilities from training data.
    oof_probabilities = cross_val_predict(
        candidate,
        X_train,
        y_train,
        cv=cv_strategy_day1,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Calculate precision and recall at different thresholds.
    precisions, recalls, thresholds = precision_recall_curve(
        y_train,
        oof_probabilities,
    )

    # Calculate F1 for each available threshold.
    f1_scores = (
        2 * precisions[:-1] * recalls[:-1]
        / (
            precisions[:-1]
            + recalls[:-1]
            + 1e-10
        )
    )

    best_index = np.argmax(f1_scores)

    day1_trials[option_name] = {
        "weight": weight,
        "threshold": thresholds[best_index],
        "precision": precisions[best_index],
        "recall": recalls[best_index],
        "f1": f1_scores[best_index],
    }

print("\nCross-validation results:")

display(
    pd.DataFrame(day1_trials).T
)

Testing: Unweighted
Testing: Balanced

Cross-validation results:


,weight,threshold,precision,recall,f1
Unweighted,NaN,0.330384,0.560568,0.739799,0.637832
Balanced,balanced,0.574173,0.557837,0.745151,0.63803


In [40]:
best_option = max(
    day1_trials,
    key=lambda name: day1_trials[name]["f1"],
)

best_config = day1_trials[best_option]

baseline_threshold = float(
    best_config["threshold"]
)

print("Selected configuration:", best_option)

print(
    "Selected threshold:",
    round(baseline_threshold, 4),
)

print(
    "Validation precision:",
    round(best_config["precision"], 4),
)

print(
    "Validation recall:",
    round(best_config["recall"], 4),
)

print(
    "Validation F1:",
    round(best_config["f1"], 4),
)

Selected configuration: Balanced
Selected threshold: 0.5742
Validation precision: 0.5578
Validation recall: 0.7452
Validation F1: 0.638


In [41]:
baseline_pipeline = make_model_pipeline(
    X_train,
    LogisticRegression(
        max_iter=2000,
        class_weight=best_config["weight"],
        random_state=RANDOM_STATE,
    ),
)

baseline_pipeline.fit(
    X_train,
    y_train,
)

print("Improved Day 1 Logistic Regression fitted.")

Improved Day 1 Logistic Regression fitted.


In [42]:
# Predict probabilities.
test_probabilities = baseline_pipeline.predict_proba(
    X_test
)[:, 1]

# Apply selected threshold.
test_predictions = (
    test_probabilities >= baseline_threshold
).astype(int)

# Calculate metrics.
day1_metrics = {
    "Model": "Day 1 Logistic Regression",
    "Accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),
    "Precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "F1": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "ROC_AUC": roc_auc_score(
        y_test,
        test_probabilities,
    ),
}

display(
    pd.DataFrame([day1_metrics]).set_index("Model")
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["No Churn", "Churn"],
        zero_division=0,
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        test_predictions,
    )
)

,Accuracy,Precision,Recall,F1,ROC_AUC
Model,,,,,
Day 1 Logistic Regression,0.762243,0.538767,0.724599,0.618016,0.841298



Classification Report:
              precision    recall  f1-score   support

    No Churn       0.89      0.78      0.83      1035
       Churn       0.54      0.72      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409


Confusion Matrix:
[[803 232]
 [103 271]]


In [43]:
day1_model_path = (
    MODEL_DIR / "churn_day1_baseline.joblib"
)

day1_artifact = {
    "pipeline": baseline_pipeline,
    "threshold": baseline_threshold,
    "model_name": "Logistic Regression",
    "positive_class": 1,
}

joblib.dump(
    day1_artifact,
    day1_model_path,
)

print("Day 1 model saved:", day1_model_path)

Day 1 model saved: C:\Users\KIIT\Desktop\RetailMind\models\churn_day1_baseline.joblib


In [44]:
def predict_churn(customer_rows, artifact=None):

    if artifact is None:
        artifact = day1_artifact

    fitted_model = artifact["pipeline"]
    threshold = artifact["threshold"]

    probabilities = fitted_model.predict_proba(
        customer_rows
    )[:, 1]

    labels = (
        probabilities >= threshold
    ).astype(int)

    return pd.DataFrame(
        {
            "predicted_churn": labels,
            "churn_probability": probabilities,
        },
        index=customer_rows.index,
    )


display(
    predict_churn(X_test.head())
)

,predicted_churn,churn_probability
437,0,0.115472
2280,1,0.853079
2235,0,0.146831
4460,1,0.660663
3761,0,0.060436


In [ ]:
project_dir = Path.cwd().resolve()
if not (project_dir / "data").is_dir() and (project_dir.parent / "data").is_dir():
    project_dir = project_dir.parent
REPORT_DIR = project_dir / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

day1_report_path = (
    REPORT_DIR / "day1_baseline_metrics.csv"
)

pd.DataFrame([day1_metrics]).to_csv(
    day1_report_path,
    index=False,
)

print("Metrics saved:", day1_report_path)